In [1]:
source setup.tcl
set year 2016
set day 23
aoc::get-puzzle $year $day 1
aoc::get-puzzle $year $day 2

set input [string trim [aoc::get-input $year $day]];
jupyter::display text/plain [string range $input 0 100]...;

namespace eval ops {
    proc cpy {arg1 target} {
        if {$arg1 in {a b c d}} {
            set ::state($target) $::state($arg1)
        } else {
            set ::state($target) $arg1
        }
    }
    proc jnz {arg1 arg2} {
        if {$arg1 in {a b c d}} {
            set val $::state($arg1)
        } else {
            set val $arg1
        }
        if {$arg2 in {a b c d}} {
            set val2 $::state($arg2)
        } else {
            set val2 $arg2
        }
        if {$val != 0} {
                ::incr ::state(pc) $val2
                # offset the incr of pc after every command
                incr ::state(pc) -1

        }
        
    }
    
    proc tgl {arg1} {
        set offset [expr {$::state(pc) + $::state($arg1)}]
        # puts tgl-$offset
        set oldcmd [lindex $::state(prog) $offset]
        switch [llength $oldcmd] {
            2 { 
                lassign $oldcmd op1 arg1
                if {$op1 eq "inc"} {
                    set op1 dec
                } else {
                    set op1 inc
                }
                lset ::state(prog) $offset [list $op1 $arg1]
            }
            3 { 
                lassign $oldcmd op1 arg1 arg2
                if {$op1 eq "jnz"} {
                    set op1 cpy
                } else {
                    set op1 jnz
                }
                lset ::state(prog) $offset [list $op1 $arg1 $arg2]
            }
        }

    }
    proc inc {reg} {
       ::incr ::state($reg)
    }
    proc dec {reg} {
        ::incr ::state($reg) -1
    }
}

proc run {input a b c d} {
    set ::state(pc) 0
    set ::state(prog) [split $input \n]
    set ::state(toggled) {}
    set ::state(a) $a
    set ::state(b) $b
    set ::state(c) $c
    set ::state(d) $d
    set visited {}
    set ::state(max) [llength $::state(prog)]
    while {$::state(pc) < $::state(max)} {
        if {$::state(pc) ni $visited} {
            lappend visited $::state(pc)
            # puts $visited
        }
        # puts ========
        set pc $::state(pc)
        set prog $::state(prog)
        # peephole optimisation
        if {[lrange $prog $pc $pc+5] eq [split {cpy b c
inc a
dec c
jnz c -2
dec d
jnz d -5} \n]} {
            # puts "Adding b*d to a"
            set b $::state(b)
            set d $::state(d)
            incr ::state(a) [expr {$b*$d}]
            set ::state(c) 0
            set ::state(d) 0
            incr ::state(pc) 5
            # puts "current command [lindex $::state(prog) $::state(pc)]"
            continue
        }
        set cmd [lindex $::state(prog) $::state(pc)]
        # parray ::state
        # puts "cmd: $::state(pc)"
        namespace eval ops [lindex $::state(prog) $::state(pc)]
        incr ::state(pc)
    }
    return $::state(a)
}

(cached)

--- Day 23: Safe Cracking --- This is one of the top floors of the nicest tower in EBHQ. The Easter Bunny's private office is here, complete with a safe hidden behind a painting, and who wouldn't hide a star in a safe behind a painting? The safe has a digital screen and keypad for code entry. A sticky note attached to the safe has a password hint on it: "eggs". The painting is of a large rabbit coloring some eggs. You see 7 . When you go to type the code, though, nothing appears on the display; instead, the keypad comes apart in your hands, apparently having been smashed. Behind it is some kind of socket - one that matches a connector in your prototype computer ! You pull apart the smashed keypad and extract the logic circuit, plug it into your computer, and plug your computer into the safe. Now, you just need to figure out what output the keypad would have sent to the safe. You extract the assembunny code from the logic chip (your puzzle input). The code looks like it uses almost the same architecture and instruction set that the monorail computer used! You should be able to use the same assembunny interpreter for this as you did there, but with one new instruction: 
 tgl x toggles the instruction x away (pointing at instructions like jnz does: positive means forward; negative means backward): 
 For one-argument instructions, inc becomes dec , and all other one-argument instructions become inc . For two-argument instructions, jnz becomes cpy , and all other two-instructions become jnz . The arguments of a toggled instruction are not affected . If an attempt is made to toggle an instruction outside the program, nothing happens . If toggling produces an invalid instruction (like cpy 1 2 ) and an attempt is later made to execute that instruction, skip it instead . If tgl toggles itself (for example, if a is 0 , tgl a would target itself and become inc a ), the resulting instruction is not executed until the next time it is reached. 
 For example, given this program: cpy 2 a
tgl a
tgl a
tgl a
cpy 1 a
dec a
dec a
 
 
 cpy 2 a initializes register a to 2 . The first tgl a toggles an instruction a ( 2 ) away from it, which changes the third tgl a into inc a . The second tgl a also modifies an instruction 2 away from it, which changes the cpy 1 a into jnz 1 a . The fourth line, which is now inc a , increments a to 3 . Finally, the fifth line, which is now jnz 1 a , jumps a ( 3 ) instructions ahead, skipping the dec a instructions. 
 In this example, the final value in register a is 3 . The rest of the electronics seem to place the keypad entry (the number of eggs, 7 ) in register a , run the code, and then send the value left in register a to the safe. 
 What value should be sent to the safe?

(cached)

--- Part Two --- The safe doesn't open, but it does make several angry noises to express its frustration. You're quite sure your logic is working correctly, so the only other thing is... you check the painting again. As it turns out, colored eggs are still eggs. Now you count 12 . As you run the program with this new input, the prototype computer begins to overheat . You wonder what's taking so long, and whether the lack of any instruction more powerful than "add one" has anything to do with it. Don't bunnies usually multiply ? Anyway, what value should actually be sent to the safe?

(cached)

cpy a b
dec b
cpy a d
cpy 0 a
cpy b c
inc a
dec c
jnz c -2
dec d
jnz d -5
dec b
cpy b c
cpy c d
dec d...

In [2]:
proc part1 input {run $input 7 0 0 0}

In [3]:
proc part2 input {run $input 12 0 0 0}

# Note

This code in the input is calculating `reg(a)!+6375`

In [4]:
aoc::results

Part1	11415 (91598 us)
Part2	479007975 (50992 us)


11415 479007975